# Retrieval-Augmented Generation (RAG)

This ipython notebook will show how to implement and use Retrieval-Augmented Generation (RAG) for question answering tasks, with a focus on leveraging external knowledge sources to improve the performance of language models.


## Step 1: Install Required Libraries and load pages from the PDF files

In [ ]:
from pathlib import Path
from pypdf import PdfReader
import numpy as np
from dotenv import load_dotenv
from google import genai
import json
from google.genai import types
import time
import chromadb
from datetime import datetime




In [ ]:
load_dotenv()
client = genai.Client()

In [ ]:
def load_pages(folder="data"):
    pages = []
    for pdf in sorted(Path(folder).glob("*.pdf")):
        reader = PdfReader(pdf)
        for i, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            pages.append({"source": pdf.name, "page": i, "text": text})
    return pages

pages = load_pages()
print(len(pages), "pages")

## Step 2: Divide the obtained text into chunks of similar sizes

In [ ]:
def chunk_text(text, size=800, overlap=100):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            # avoid cutting in the middle of a word
            last_space = text.rfind(" ", start, end)
            if last_space > start + size // 2:
                end = last_space
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start = end - overlap
    return chunks


def build_chunks(pages, size=800, overlap=100, min_len=100):
    chunks = []
    for p in pages:
        for n, piece in enumerate(chunk_text(p["text"], size, overlap), start=1):
            if len(piece) < min_len:
                continue
            chunks.append({
                "id": f'{p["source"]}-p{p["page"]}-c{n}',
                "text": piece,
                "source": p["source"],
                "page": p["page"],
            })
    return chunks


chunks = build_chunks(pages)
print(len(chunks), "chunks")
assert len({c["id"] for c in chunks}) == len(chunks), "duplicate ids"

## Step 3: Sanity Check

Check if the generated chunks picked at random make sense to read, as well to check if the overlap is working.

In [ ]:
import random
for c in random.sample(chunks, 5):
    print(c["id"], len(c["text"]))
    print(c["text"], "\n---")

In [ ]:
for x, y in zip(chunks, chunks[1:]):
    if x["source"] == y["source"] and x["page"] == y["page"]:
        print(x["id"], y["id"])
        print(repr(x["text"][-120:]))
        print(repr(y["text"][:120]))
        print("overlap found:", y["text"][:60] in x["text"])
        break

## Step 4: Store the chunks in a JSON file


In [ ]:
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

## Step 5: Generate Embeddings and Store in ChromaDB

In [ ]:

EMBEDDING_MODEL = "gemini-embedding-001"

def get_embeddings(texts, task="RETRIEVAL_DOCUMENT") -> np.ndarray:
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task),
    )
    assert len(response.embeddings) == len(texts), "embedding count mismatch"
    return np.array([e.values for e in response.embeddings])

vecs = get_embeddings([c["text"] for c in chunks[:3]])
print(vecs.shape)   # (3, vector_length)

In [ ]:
db = chromadb.PersistentClient(path="chroma_db")
col = db.get_or_create_collection(name="docs", metadata={"hnsw:space": "cosine"})
print(col.count())

In [ ]:
def embed_and_store(chunks, col, batch_size=50, per_minute=90):
    done = set(col.get(ids=[c["id"] for c in chunks])["ids"])
    todo = [c for c in chunks if c["id"] not in done]
    print(f"{len(done)} already stored, {len(todo)} to embed")

    for i in range(0, len(todo), batch_size):
        batch = todo[i:i + batch_size]
        for attempt in range(4):
            try:
                vecs = get_embeddings([c["text"] for c in batch]).tolist()
                break
            except Exception as e:
                wait = 20 * (attempt + 1)
                print(f"failed: {str(e)[:100]} | retrying in {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError("batch failed after 4 attempts")

        col.upsert(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            embeddings=vecs,
            metadatas=[{"source": c["source"], "page": c["page"]} for c in batch],
        )
        print(f"stored {min(i + batch_size, len(todo))}/{len(todo)}")
        time.sleep(60 * len(batch) / per_minute)   # about 33s for 50 texts

embed_and_store(chunks, col)
print(col.count(), "items in collection")

# Step 6: Query Testing


In [ ]:
def retrieve(question, k=4):
    qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
    res = col.query(query_embeddings=[qv], n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

for doc, meta, dist in retrieve("O que é uma ferida crónica?"):
    print(round(dist, 3), meta)
    print(doc[:200], "\n---")


#Questions asked
## What kind of political system does Portugal have?
## What is a chronic wound?
## Who is Ryze?
## How do I bake a cake?
## O que é uma ferida crónica?


In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.5-flash-lite"
LOG_FILE = "retrieval_logs.jsonl"

# 1. Function retrieve(question, k=4) that generates the embedding, prints and returns the chunks with distances.
def retrieve(question: str, k: int = 4):
    qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
    res = col.query(
        query_embeddings=[qv],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    chunks_retrieved = []
    print(f"\n=== Chunks Retrieved for: '{question}' ===")
    for cid, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        item = {
            "id": cid,
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "distance": round(float(dist), 4),
        }
        chunks_retrieved.append(item)
        print(f"[{item['distance']:.3f}] {item['source']} (page. {item['page']}) | ID: {item['id']}")
        print(f"{item['text'][:200]}...\n---")
        
    return chunks_retrieved

# 2. Auxiliar Function to register the chunks retrieved for each call
def log_call(question: str, chunks: list, answer: str):
    record = {
        "timestamp": datetime.now().isoformat(),
        "question": question,
        "retrieved_chunks": chunks,
        "answer": answer,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# 3. Generation function with strict prompt, low temperature and returns the sources as well
def generate_answer(question: str, k: int = 4):
    chunks = retrieve(question, k=k)
    
    # Format the context with the source
    blocks_context = []
    for c in chunks:
        blocks_context.append(
            f"[File: {c['source']}, Page: {c['page']}]\n{c['text']}"
        )
    context_str = "\n\n---\n\n".join(blocks_context)
    
    prompt_system = (
        "You're a rigurous assistant of questions and answers.\n"
        "Mandatory Rules:\n"
        "1. Only answer based on the context given below.\n"
        "2. If the answer isn't explicit in the context, answer with I don't know."
        "(ex: 'I can't answer based on the given documents.'). Do not create new information nor use external knowledge.\n"
        "3. Always cite the file and the page from where you got the information in the answer's body."
        "(format: [File: name.pdf, Page: X])."
    )
    
    prompt_user = f"CONTEXTO:\n{context_str}\n\nQUESTION: {question}"
    
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt_user,
        config=types.GenerateContentConfig(
            system_instruction=prompt_system,
            temperature=0.1,
        ),
    )
    
    text_answer = response.text.strip()
    
    # Extract unique line of consulted sources
    sources = sorted({f"{c['source']} (pág. {c['page']})" for c in chunks})
    final_answer = f"{text_answer}\n\nFontes recuperadas:\n" + "\n".join(f"- {f}" for f in sources)
    
    # Register the chunks and the answers for debugging
    log_call(question, chunks, text_answer)
    
    return {
        "answer": text_answer,
        "sources": sources,
        "chunks": chunks,
        "formatted_output": final_answer,
    }

#Test
result = generate_answer("What ingredients and oven temperature are needed to bake a traditional chocolate cake?", k=4)
print("\n=== Final Answer ===")
print(result["formatted_output"])

# Step 7: Improving the RAG

Given that the Query 3 is failing to succeed, this step has the purpose of trying to understand what's going on and what's going wrong.

In [ ]:
SRC = "Automated_Dressing_Saturation_Analysis.pdf"
def find(substr):
    res = col.get(
        where={"source": SRC},
        where_document={"$contains": substr},
        include=["documents", "metadatas"],
    )
    return {i: (d, m) for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])}

# dataset size: chunks with 459 AND a marker that makes them actually state the dataset composition
size_markers = ["281 real", "In total it contains", "out of 459", "154 positives"]
gold_size = {i for i, (d, m) in find("459").items() if any(k in d for k in size_markers)}

# best pipeline: 92.71 together with the recall 93.88 (abstract, results p.50, Table 4.19)
gold_pipe = {i for i, (d, m) in find("92.71").items() if "93.88" in d}

for name, group in [("DATASET SIZE", gold_size), ("BEST PIPELINE", gold_pipe)]:
    print(f"\n== {name} ({len(group)} chunks) ==")
    allhits = {**find("459"), **find("92.71")}
    for i in sorted(group):
        d, m = allhits[i]
        print(i, "| page", m["page"], "|", d[:90].replace("\n", " "))

### Gold chunks (ground truth for the diagnosis)

A *gold chunk* is a chunk that really contains the answer. I found them with an exact text search (`col.get(where_document={"$contains": ...})`), not with embeddings, so they do not depend on the retrieval being tested.

- **Dataset size:** chunks containing `459` **and** a marker that states the dataset composition (`281 real`, `In total it contains`, `out of 459`, `154 positives`). 5 chunks: Section 3.3.1 and the class-imbalance paragraph (page 39), the Limitations paragraph (page 81) and the two Conclusions chunks (page 82).
- **Best pipeline:** chunks containing both `92.71` and `93.88`. 5 chunks: the Portuguese and English abstracts (pages 4 and 5), the Results prose and Table 4.19 (page 66) and Table 4.32 (page 79).
- **Excluded on purpose:** the reference-list hit (`pp. 459–479`, a false positive for `459`) and the Table 3.1 row `Total 459`, which has no context around it.

**Page numbers:** the `page` stored in Chroma is the PDF page index. In the body of the dissertation the printed page is the PDF page minus 16 (PDF page 39 is printed page 23, PDF page 66 is printed page 50). In the front matter the offset is different (roman numerals), so it is not constant. The eval set has to use PDF pages or chunk ids, not the printed numbers.

In [ ]:
_cache = {}

def rank_all(question):
    """All chunks ranked by distance for this question (embedding cached)."""
    if question not in _cache:
        qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
        res = col.query(
            query_embeddings=[qv],
            n_results=col.count(),
            include=["distances", "metadatas"],
        )
        _cache[question] = list(zip(res["ids"][0], res["distances"][0], res["metadatas"][0]))
    return _cache[question]

def rank_report(label, question, gold, show_top=5):
    ranked = rank_all(question)
    ids = [r[0] for r in ranked]
    pos = sorted(ids.index(g) + 1 for g in gold if g in ids)   # 1-based ranks of gold chunks
    best = pos[0] if pos else None
    best_dist = ranked[best - 1][1] if best else None
    print(f"\n### {label}")
    print(f"gold ranks: {pos}")
    print(f"best rank: {best} (distance {best_dist:.3f})" if best else "best rank: none")
    for k in (4, 8, 10, 20):
        n = sum(1 for p in pos if p <= k)
        print(f"  top-{k}: {n}/{len(gold)} gold chunks")
    print("  distance at rank 1/4/10/20:", [round(ranked[r - 1][1], 3) for r in (1, 4, 10, 20)])
    print(f"  top {show_top}:")
    for r, (cid, dist, meta) in enumerate(ranked[:show_top], start=1):
        tag = " <-- GOLD" if cid in gold else ""
        print(f"   {r}. {dist:.3f} {cid} (page {meta['page']}){tag}")

In [ ]:
Q_FULL = ("How many images are in the dissertation's dataset, and which pipeline "
          "achieved the highest overall F-beta score on the fold 0 test set?")
Q_A = "How many images are in the dissertation's dataset?"
Q_B = "Which pipeline achieved the highest overall F-beta score on the fold 0 test set?"

rank_report("FULL question vs dataset-size gold", Q_FULL, gold_size)
rank_report("FULL question vs best-pipeline gold", Q_FULL, gold_pipe)
rank_report("Sub-question A vs dataset-size gold", Q_A, gold_size)
rank_report("Sub-question B vs best-pipeline gold", Q_B, gold_pipe)


### Q3 diagnosis: results

Gold ranks are positions among all 489 chunks (1 = closest). The lists are sorted, so this output shows *how many* gold chunks sit where, but not *which* chunk is which rank.

| Query | Gold group | Gold ranks | Gold in top 4 / 10 / 20 |
|---|---|---|---|
| Full question (`Q_FULL`) | dataset size | 30, 34, 67, 74, 137 | 0 / 0 / 0 |
| Full question (`Q_FULL`) | best pipeline | 2, 9, 13, 139, 152 | 1 / 2 / 3 |
| Sub-question A (dataset size) | dataset size | 1, 9, 12, 13, 25 | 1 / 2 / 4 |
| Sub-question B (best pipeline) | best pipeline | 19, 20, 75, 117, 140 | 0 / 0 / 2 |

**What the numbers say**

1. **The dataset-size half is a retrieval failure.** For the full question the best dataset-size chunk is at rank 30. A larger `k` cannot realistically fix this: `k=20` still finds 0 of 5, and `k=30` would put about 6% of the whole collection into the prompt.
2. **The F-beta half was not a retrieval failure.** An abstract chunk with 92.71% and 93.88% was already at rank 2, inside the top 4 of the original run. The original answer was still "I can't answer", so that refusal came from the prompt rule ("if the answer isn't explicit, I don't know"), which refused the whole question because one half was missing. Q3 therefore had two causes at once: a retrieval problem for one half and a prompt problem for the other.
3. **Blending hurts the dataset-size half.** The best dataset-size chunk moves from rank 30 (full question) to rank 1 (sub-question A). The likely reason is that one embedding for two topics sits between both targets, but I did not test that directly.
4. **Sub-question B alone is worse than the full question** for the best-pipeline gold (best rank 19 against 2). I did not establish why. Two hypotheses to test: many chunks discuss F-beta scores of the other approaches, so the topic is crowded; or the full question has extra words (`dataset`, `dissertation`) that pull toward the abstract. This matters for decomposition: if only the sub-queries are used with a small `k` per query, the F-beta half can get lost.
5. **The distance curve is flat.** Rank 1 to rank 20 spans only about 0.03 for the full question (0.231 to 0.263) and for B (0.244 to 0.276), and about 0.06 for A (0.226 to 0.283). Ranks are volatile with such small gaps, and a distance threshold cannot separate good from bad chunks here.
6. **Front matter shows up, but was not the main cause for Q3.** In the later `k=8` run a List of Tables chunk (page 14) is retrieved at position 6, which supports cleaning the corpus, but the abstract chunk was still ranked above it.

# Step 8: Adding query decomposition and partial answering to the RAG

**Plan for Step 8.** Q3 has two stacked causes (Step 7), so it needs two kinds of fix:

1. **Prompt (generation):** rule 3 now allows partial answers: answer what the context supports and state which part cannot be answered. "I don't know" stays for questions where nothing relevant appears.
2. **Retrieval:** `k=8` as a first attempt, then `decompose_query` (Gemini splits the question into single-topic sub-queries, with structured output) and `retrieve_multi` (retrieve per sub-query and merge without duplicates).

Because of finding 4 in Step 7, the merge should probably also keep the results of the **original full question**, not only of the sub-queries.

In [ ]:
from pydantic import BaseModel
import json

class DecomposedQuery(BaseModel):
    sub_queries: list[str]

def decompose_query(question: str) -> list[str]:
    system_instruction = (
        "You are a search query optimizer. Break the user's complex, multi-part question "
        "into independent, single-topic questions that are optimized for semantic vector search. "
        "If the question is already simple and single-topic, return a list containing just that question."
    )
    
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=DecomposedQuery,
        ),
    )
    
    return json.loads(response.text)["sub_queries"]

# Example usage:
# sub_queries = decompose_query("How many images are in the dataset and which pipeline won on fold 0?")
# Returns: ["How many images are in the dataset?", "Which pipeline won on fold 0?"]

**Status of the changes in the next cell:**

- Partial-answer prompt (rule 3): active in `generate_answer`.
- `retrieve_multi` and `decompose_query`.

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.5-flash-lite"
LOG_FILE = "retrieval_logs.jsonl"

# 1. Function retrieve(question, k=4) that generates the embedding, prints and returns the chunks with distances.
def retrieve(question: str, k: int = 4):
    qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
    res = col.query(
        query_embeddings=[qv],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    chunks_retrieved = []
    print(f"\n=== Chunks Retrieved for: '{question}' ===")
    for cid, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        item = {
            "id": cid,
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "distance": round(float(dist), 4),
        }
        chunks_retrieved.append(item)
        print(f"[{item['distance']:.3f}] {item['source']} (page. {item['page']}) | ID: {item['id']}")
        print(f"{item['text'][:200]}...\n---")
        
    return chunks_retrieved

# 2. Auxiliar Function to register the chunks retrieved for each call
def log_call(question: str, chunks: list, answer: str):
    record = {
        "timestamp": datetime.now().isoformat(),
        "question": question,
        "retrieved_chunks": chunks,
        "answer": answer,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# 3. NEW retrieval multi-query assistant
def retrieve_multi(question: str, k_per_query: int = 3) -> list[dict]:
    sub_queries = decompose_query(question)
    print(f"\n[Query Decomposition] Original: '{question}'")
    print(f"[Query Decomposition] Sub-queries: {sub_queries}")
    
    all_chunks = {}
    
    for sub_q in sub_queries:
        # Calls your existing retrieve() function
        chunks = retrieve(sub_q, k=k_per_query) 
        for c in chunks:
            # Deduplicate by chunk ID so the LLM doesn't read the same text twice
            if c['id'] not in all_chunks:
                all_chunks[c['id']] = c
                
    final_chunks = list(all_chunks.values())
    print(f"\n[Query Decomposition] Total unique chunks retrieved: {len(final_chunks)}\n")
    
    return final_chunks

# 4. Generation function with strict prompt, low temperature and returns the sources as well
def generate_answer(question: str, k: int = 4):
    chunks = retrieve_multi(question, k_per_query=k)
    
    # Format the context with the source
    blocks_context = []
    for c in chunks:
        blocks_context.append(
            f"[File: {c['source']}, Page: {c['page']}]\n{c['text']}"
        )
    context_str = "\n\n---\n\n".join(blocks_context)
    
    prompt_system = (
        "You're a rigorous assistant of questions and answers.\n"
        "Mandatory Rules:\n"
        "1. Only answer based on the context given below.\n"
        "2. If the answer isn't explicit in the context, answer with I don't know "
        "(ex: 'I can't answer based on the given documents.'). Do not create new information nor use external knowledge.\n"
        "3. You are allowed to give partial answers. If the context only supports part of a multi-part question, answer what you can and explicitly state which part of the question you cannot answer based on the documents.\n"
        "4. Always cite the file and the page from where you got the information in the answer's body "
        "(format: [File: name.pdf, Page: X])."
    )
    
    prompt_user = f"CONTEXTO:\n{context_str}\n\nQUESTION: {question}"
    
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt_user,
        config=types.GenerateContentConfig(
            system_instruction=prompt_system,
            temperature=0.1,
        ),
    )
    
    text_answer = response.text.strip()
    
    # Extract unique line of consulted sources
    sources = sorted({f"{c['source']} (pág. {c['page']})" for c in chunks})
    final_answer = f"{text_answer}\n\nFontes recuperadas:\n" + "\n".join(f"- {f}" for f in sources)
    
    # Register the chunks and the answers for debugging
    log_call(question, chunks, text_answer)
    
    return {
        "answer": text_answer,
        "sources": sources,
        "chunks": chunks,
        "formatted_output": final_answer,
    }

#Test
result = generate_answer("How many images are in the dissertation's dataset, and which pipeline achieved the highest overall F-beta score on the fold 0 test set?", k=8)
print("\n=== Final Answer ===")
print(result["formatted_output"])

### Result: Q3 with the partial-answer prompt and `k=8`

The stored output of the cell above is from a run on Q3 with `k=8` (the code in the cell now tests another question, so re-run Q3 before committing if you want the notebook to be reproducible). What it shows:

- **Partial answer works as intended.** The model answers the total number of images in the dataset and states that it cannot answer the highest F-beta. 
- **The dataset-size chunk is still out of reach** (best rank 30 for the full question), so `k=8` cannot fix it. This is the half that needs decomposition.